# Block 3 lecture — joins and grain

**The lecture's queries, as Eduardo ran them: for review after class. Nothing here is typed in class.**

The cell numbers match the slides ("cell 1" to "cell 11"). Run the two setup cells first, then any numbered cell.
Before you run one, say what you expect it to return: that is the habit the lecture practised.

This notebook uses three tables: `orders`, `order_items`, `customers`. The lab does not need it. At the end: what to
remember from Block 3, and the common mistakes.

In [ ]:
import os
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_rows", 400)   # show every row of a result: a census is never "the top few"

# Anchor to the project folder (DS1, Block 1), then stand there: every path below is from the project folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
print("Working in:", Path.cwd())   # must print this project's folder

con = duckdb.connect()             # an in-memory database; it reads the files in data/raw/ directly

In [ ]:
# Every table as a view with its file's name, so a query can say FROM orders instead of the path.
# A view is a saved query (Lab 1): nothing is copied; the file in data/raw/ is read each time.
TABLES = ["orders", "order_items", "customers"]
for t in TABLES:
    con.sql(f"CREATE OR REPLACE VIEW {t} AS SELECT * FROM 'data/raw/{t}.csv'")
print("views:", ", ".join(TABLES))

### 1. The three counts: `orders` joined to `order_items`

In [ ]:
con.sql("""
    SELECT (SELECT COUNT(*) FROM orders) AS rows_in_orders,
           COUNT(*)                      AS rows_in_result,
           COUNT(DISTINCT o.order_id)    AS distinct_orders_in_result
    FROM orders AS o
    JOIN order_items AS i ON i.order_id = o.order_id
""").df()

### 2. The same join as a `LEFT JOIN`

In [ ]:
con.sql("""
    SELECT (SELECT COUNT(*) FROM orders) AS rows_in_orders,
           COUNT(*)                      AS rows_in_result,
           COUNT(DISTINCT o.order_id)    AS distinct_orders_in_result
    FROM orders AS o
    LEFT JOIN order_items AS i ON i.order_id = o.order_id
""").df()

### 3. The anti-join: orders with no items

In [ ]:
con.sql("""
    SELECT COUNT(*) AS orders_with_no_items
    FROM orders AS o
    LEFT JOIN order_items AS i ON i.order_id = o.order_id
    WHERE i.order_id IS NULL
""").df()

### 4. What are they? The orders with no items, by status

In [ ]:
con.sql("""
    SELECT o.order_status, COUNT(*) AS orders
    FROM orders AS o
    LEFT JOIN order_items AS i ON i.order_id = o.order_id
    WHERE i.order_id IS NULL
    GROUP BY o.order_status
    ORDER BY orders DESC
""").df()

### 5. One order's items: the orange order in the picture

In [ ]:
con.sql("""
    SELECT order_id, order_item_id, product_id, price
    FROM order_items
    WHERE order_id LIKE '181ff95f%'
    ORDER BY order_item_id
""").df()

### 6. Orders per customer state

In [ ]:
con.sql("""
    SELECT c.customer_state, COUNT(*) AS orders
    FROM orders AS o
    JOIN customers AS c ON c.customer_id = o.customer_id
    GROUP BY c.customer_state
    ORDER BY orders DESC
    LIMIT 5
""").df()

### 7. The same count, after joining `order_items`

In [ ]:
con.sql("""
    SELECT c.customer_state, COUNT(*) AS orders
    FROM orders AS o
    JOIN customers AS c ON c.customer_id = o.customer_id
    JOIN order_items AS i ON i.order_id = o.order_id
    GROUP BY c.customer_state
    ORDER BY orders DESC
    LIMIT 5
""").df()

### 8. `customers`: one row is one what?

In [ ]:
con.sql("""
    SELECT COUNT(*)                           AS n_rows,
           COUNT(DISTINCT customer_id)        AS n_customer_id,
           COUNT(DISTINCT customer_unique_id) AS n_customer_unique_id
    FROM customers
""").df()

### 9. Pattern (a): items sold per seller, no join

In [ ]:
con.sql("""
    SELECT seller_id, COUNT(*) AS items
    FROM order_items
    GROUP BY seller_id
    ORDER BY items DESC, seller_id
    LIMIT 5
""").df()

### 10. Pattern (b): items per order, by customer state

In [ ]:
con.sql("""
    WITH order_size AS (
        SELECT order_id, COUNT(*) AS n_items
        FROM order_items
        GROUP BY order_id
    )
    SELECT c.customer_state, COUNT(*) AS orders, ROUND(AVG(s.n_items), 2) AS items_per_order
    FROM order_size AS s
    JOIN orders AS o    ON o.order_id = s.order_id
    JOIN customers AS c ON c.customer_id = o.customer_id
    GROUP BY c.customer_state
    ORDER BY orders DESC
    LIMIT 5
""").df()

### 11. The three counts for that join

In [ ]:
con.sql("""
    WITH order_size AS (SELECT order_id, COUNT(*) AS n_items FROM order_items GROUP BY order_id)
    SELECT (SELECT COUNT(*) FROM order_size) AS rows_going_in,
           COUNT(*)                          AS rows_coming_out,
           COUNT(DISTINCT s.order_id)        AS distinct_orders_out
    FROM order_size AS s
    JOIN orders AS o    ON o.order_id = s.order_id
    JOIN customers AS c ON c.customer_id = o.customer_id
""").df()

## What to remember from Block 3

1. **A join is a claim about how two grains relate.** After every join, run three counts: the rows in the left
   table, the rows in the result, and `COUNT(DISTINCT left_key)` in the result. Say what you expect before you run
   them.
2. **Read the gaps between the counts.** More rows in the result than distinct keys: some left rows matched several
   rows, and the result is multiplied. Fewer distinct keys than rows in the left table: some left rows matched
   nothing. All three equal: every left row matched exactly once. For `orders` joined to `order_items` the counts
   were 2,653 · 3,024 · 2,610 — both things at once.
3. **`INNER` against `LEFT`.** A plain `JOIN` (an `INNER JOIN`) keeps only the rows that matched and drops the rest
   without a word. A `LEFT JOIN` keeps every row of the left table and fills the right side with NULL. `RIGHT JOIN`
   keeps every row of the right table, `FULL JOIN` every row of both; write a `LEFT JOIN` from the side you mean.
4. **The anti-join finds what a join lost:** `LEFT JOIN … WHERE right_table.key IS NULL`. In this file, 43 orders
   have no items (25 canceled, 17 unavailable, 1 invoiced). Whether they belong in your number is a question about
   the business; the anti-join is how you find out that you have to ask it.
5. **Fan-out is silent.** A left row with several matches appears once per match, and everything carried along from
   its side is repeated. "Orders per state" after joining `order_items` counts items: São Paulo's 1,133 orders
   became 1,278 rows (17 orders with no items dropped, 162 extra rows added). The query runs and nothing warns you;
   only the counts show it. **Multiplication happens when a left row has several matches** — a join does not
   multiply by itself.
6. **A table's name is not its grain.** `customers` has one row per order: `customer_id` is new for every order
   (2,653 of them), and the person is `customer_unique_id` (2,634). So `orders` to `customers` finds exactly one
   row each time and cannot multiply, and counting people needs `customer_unique_id`.
7. **Two patterns.** (a) The dimension is on the measure's own table: no join (items per seller is `order_items`
   alone). Unnecessary joins add dependencies. (b) The dimension is on a parent: aggregate to the parent's grain
   first, in a CTE (`WITH name AS (…)`), then join where each row finds exactly one match. The three counts prove it:
   2,610 · 2,610 · 2,610.
8. **"Aggregate, then join" is not a cure.** It is pattern (b). On a pattern (a) question it throws the dimension
   away, and an order-level total joined back to its items is repeated once per item. `COUNT(DISTINCT …)` hides a
   fan-out instead of fixing it; `SUM(DISTINCT …)` adds up distinct *values*, not distinct rows.
9. **Direction is a decision, and the counts go in the commit message.** `a LEFT JOIN b` keeps every row of `a`;
   `b LEFT JOIN a` keeps every row of `b`. Decide whose rows the question is about, and start from that table. Any
   commit that changes a join carries its three counts: `git commit -m "what you did" -m "the three counts"`.

## Common mistakes

- **Trusting a total because it looks plausible.** One error can push a number down while another pushes it up:
  São Paulo's 1,278 "orders" were 17 dropped orders and 162 repeated rows. Run the three counts, not just the total.
- **Patching a fan-out with `DISTINCT`.** `COUNT(DISTINCT o.order_id)` makes the order count right after joining
  `order_items`, but every other column in the result is still repeated. `SUM(DISTINCT i.price)` adds up distinct
  prices: the three items of order `181ff95f…` all cost 26.90, and it counts 26.90 once. Fix the grain instead.
- **Counting rows and calling them orders.** After a join, `COUNT(*)` counts the rows of the result: items, if you
  joined `order_items`. Say what one row of the result is before you count it.
- **An anti-join with a plain `JOIN`, or with `= NULL`.** A plain `JOIN` has already thrown the unmatched rows away,
  so `WHERE i.order_id IS NULL` after it returns 0. `WHERE i.order_id = NULL` is never true. Use `LEFT JOIN … WHERE
  i.order_id IS NULL`.
- **Counting people with the wrong key.** "Orders per customer" grouped by `customer_id` is 1 for everyone, because
  `customer_id` is one per order. The person is `customer_unique_id`.
- **Carrying a parent's number down to its children.** An order-level number joined to that order's items is
  repeated once per item, and a `SUM` of it counts a three-item order three times. Aggregate the children to the
  parent's grain first, then join.